In [4]:
# ============================================================
# Multi-Task (Option A): Alternating-task training
# Shared ESM-2 backbone + 2 heads:
#   - ΔΔG regression head on (wt, mut) pairs
#   - Allergenicity classification head on single sequences
# FIRST VERSION: no attention / no attributions
# ============================================================

# ── Imports ─────────────────────────────────────────────────
import os, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr, spearmanr
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import esm
from datetime import datetime

# Optional (nice-to-have) metrics for classification
try:
    from sklearn.metrics import roc_auc_score, accuracy_score
    _HAS_SKLEARN = True
except Exception:
    _HAS_SKLEARN = False

# ── Setup ───────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
random.seed(42); np.random.seed(42); torch.manual_seed(42)
print(f"Device: {DEVICE}")

CFG = dict(
    # ΔΔG (pair) data
    ddg_train_csv  = "project_data/mega_train.csv",
    ddg_val_csv    = "project_data/mega_val.csv",
    ddg_label_col  = "ddG_ML",

    # Allergen (single sequence) data  (YOU MUST PROVIDE RAW SEQUENCES, not 1280-d embeddings)
    # Expected columns: "seq" (or change below), "label" (0/1), optional "id"
    alg_train_csv = "project_data/algpred2_train_seq.csv",
    alg_val_csv   = "project_data/algpred2_test_seq.csv", 
    alg_seq_col    = "sequence",
    alg_label_col  = "label",
    alg_id_col     = "id",

    # Training
    epochs     = 6,
    batch_size_ddg = 1,
    batch_size_alg = 1,

    lr_head    = 1e-3,
    lr_esm     = 5e-6,
    weight_decay = 1e-2,

    patience   = 5,
    grad_clip  = 1.0,

    lr_factor  = 0.5,
    lr_patience= 2,

    # Multi-task balancing (Option A)
    # sample pattern: ["ddg","alg","ddg","alg", ...] if task_ratio=(1,1)
    task_ratio = (1, 1),          # (ddg_steps, alg_steps)
    lambda_ddg = 1.0,
    lambda_alg = 1.0,

    # Finetuning control
    unfreeze_last_n = 0,          # 0/1/2/3...
)

# ── Datasets ────────────────────────────────────────────────
class ProteinPairDataset(Dataset):
    def __init__(self, csv_path, label_col="ddG_ML"):
        df = pd.read_csv(csv_path)
        self.wt  = df["wt_seq"].tolist()
        self.mut = df["aa_seq"].tolist()
        self.y   = df[label_col].astype(float).values
        self.ids = df["name"].tolist()

    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.wt[i], self.mut[i], float(self.y[i]), self.ids[i]


class AllergenSeqDataset(Dataset):
    """
    Expects RAW sequences, not precomputed embeddings.
    Columns:
      - seq_col: protein sequence string
      - label_col: 0/1
      - id_col: optional
    """
    def __init__(self, csv_path, seq_col="seq", label_col="label", id_col="id"):
        df = pd.read_csv(csv_path)
        self.seq   = df[seq_col].astype(str).tolist()
        self.y     = df[label_col].astype(int).values
        if id_col in df.columns:
            self.ids = df[id_col].astype(str).tolist()
        else:
            self.ids = [f"seq_{i}" for i in range(len(df))]

    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.seq[i], int(self.y[i]), self.ids[i]


# ── Collates ────────────────────────────────────────────────
def make_collate_pair(batch_converter):
    def collate(batch):
        wt, mut, y, ids = zip(*batch)
        _, _, wt_tok  = batch_converter(list(zip(ids, wt)))
        _, _, mut_tok = batch_converter(list(zip(ids, mut)))
        return wt_tok, mut_tok, torch.tensor(y, dtype=torch.float32), list(ids)
    return collate


def make_collate_single(batch_converter):
    def collate(batch):
        seq, y, ids = zip(*batch)
        _, _, tok = batch_converter(list(zip(ids, seq)))
        return tok, torch.tensor(y, dtype=torch.float32), list(ids)
    return collate


# ── Model ───────────────────────────────────────────────────
class MultiTaskESM(nn.Module):
    """
    Shared ESM backbone + two heads:
      - ddg_head: uses pooled(wt), pooled(mut), diff, prod -> regression
      - alg_head: uses pooled(seq) -> logit
    """
    def __init__(self, esm_model, alphabet, hidden=256, dropout=0.2):
        super().__init__()
        self.esm, self.alphabet = esm_model, alphabet
        self.repr_layer = esm_model.num_layers
        d = esm_model.embed_dim

        # ΔΔG head (same as yours)
        self.ddg_head = nn.Sequential(
            nn.LayerNorm(4*d),
            nn.Linear(4*d, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

        # Allergen head (binary logit)
        self.alg_head = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def encode(self, tokens):
        out = self.esm(tokens, repr_layers=[self.repr_layer], return_contacts=False)
        h = out["representations"][self.repr_layer]                     # [B, L, d]
        mask = (tokens != self.alphabet.padding_idx) & (tokens != self.alphabet.eos_idx)
        mask[:, 0] = False                                              # ignore CLS
        denom = mask.sum(1, keepdim=True).float().clamp_min(1.0)
        return (h * mask.unsqueeze(-1).float()).sum(1) / denom          # [B, d]

    def forward_ddg(self, wt_tok, mut_tok):
        w = self.encode(wt_tok)
        m = self.encode(mut_tok)
        x = torch.cat([w, m, m - w, m * w], dim=-1)
        return self.ddg_head(x).squeeze(-1)                             # [B]

    def forward_alg(self, tok):
        z = self.encode(tok)
        return self.alg_head(z).squeeze(-1)                             # [B] logits


def count_params(model):
    trainable     = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"{'Module':<20} {'Params':>12} {'Trainable':>12}")
    print("-" * 46)
    for name, module in model.named_children():
        p  = sum(x.numel() for x in module.parameters())
        pt = sum(x.numel() for x in module.parameters() if x.requires_grad)
        print(f"{name:<20} {p:>12,} {pt:>12,}")
    print("-" * 46)
    print(f"{'Trainable':<20} {trainable:>12,}")
    print(f"{'Non-trainable':<20} {non_trainable:>12,}")
    print(f"{'Total':<20} {trainable+non_trainable:>12,}")
    print(f"{'Size (MB)':<20} {(trainable+non_trainable)*4/1024**2:>11.3f}")


# ── Helpers ────────────────────────────────────────────────
def freeze_and_unfreeze_esm(esm_model, unfreeze_last_n: int):
    for p in esm_model.parameters():
        p.requires_grad = False

    n = int(unfreeze_last_n)
    if n > 0:
        for block in esm_model.layers[-n:]:
            for p in block.parameters():
                p.requires_grad = True

    # unfreeze all LayerNorm params (often stabilizes finetuning)
    for m in esm_model.modules():
        if isinstance(m, nn.LayerNorm):
            for p in m.parameters():
                p.requires_grad = True


def cycle(loader):
    """Infinite loader cycling."""
    while True:
        for batch in loader:
            yield batch


# ── Train (Multi-task alternating) ──────────────────────────
def train_multitask(cfg):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_dir  = f"checkpoints_multitask_{timestamp}"
    os.makedirs(save_dir, exist_ok=True)
    best_path = os.path.join(save_dir, f"best_{timestamp}.pt")

    # Load ESM
    esm_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
    freeze_and_unfreeze_esm(esm_model, cfg.get("unfreeze_last_n", 0))

    model = MultiTaskESM(esm_model, alphabet).to(DEVICE)
    batch_converter = alphabet.get_batch_converter()

    # DataLoaders
    ddg_collate = make_collate_pair(batch_converter)
    alg_collate = make_collate_single(batch_converter)

    ddg_train = DataLoader(
        ProteinPairDataset(cfg["ddg_train_csv"], label_col=cfg["ddg_label_col"]),
        batch_size=cfg["batch_size_ddg"], shuffle=True,  collate_fn=ddg_collate
    )
    ddg_val = DataLoader(
        ProteinPairDataset(cfg["ddg_val_csv"], label_col=cfg["ddg_label_col"]),
        batch_size=cfg["batch_size_ddg"], shuffle=False, collate_fn=ddg_collate
    )

    alg_train = DataLoader(
        AllergenSeqDataset(cfg["alg_train_csv"], seq_col=cfg["alg_seq_col"],
                           label_col=cfg["alg_label_col"], id_col=cfg["alg_id_col"]),
        batch_size=cfg["batch_size_alg"], shuffle=True,  collate_fn=alg_collate
    )
    alg_val = DataLoader(
        AllergenSeqDataset(cfg["alg_val_csv"], seq_col=cfg["alg_seq_col"],
                           label_col=cfg["alg_label_col"], id_col=cfg["alg_id_col"]),
        batch_size=cfg["batch_size_alg"], shuffle=False, collate_fn=alg_collate
    )

    # Optimizer: head vs esm LR
    opt = torch.optim.AdamW(
        [
            {"params": [p for n,p in model.named_parameters() if p.requires_grad and not n.startswith("esm.")], "lr": cfg["lr_head"]},
            {"params": [p for n,p in model.named_parameters() if p.requires_grad and     n.startswith("esm.")], "lr": cfg["lr_esm"]},
        ],
        weight_decay=cfg.get("weight_decay", 1e-2)
    )

    # Scheduler: we need ONE scalar to monitor; choose ddg_val_rmse as primary
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min",
        factor=cfg.get("lr_factor", 0.5),
        patience=cfg.get("lr_patience", 2),
    )

    # Losses
    mse = nn.MSELoss(reduction="mean")
    bce = nn.BCEWithLogitsLoss(reduction="mean")

    count_params(model)

    history = {
        "ddg_train_rmse": [], "ddg_val_rmse": [], "ddg_pearson": [], "ddg_spearman": [],
        "alg_val_auc": [], "alg_val_acc": []
    }

    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_rmse = float("inf")
    patience_left = cfg["patience"]

    # Build the alternating task schedule
    ddg_steps, alg_steps = cfg.get("task_ratio", (1, 1))
    task_schedule = (["ddg"] * int(ddg_steps)) + (["alg"] * int(alg_steps))
    if len(task_schedule) == 0:
        task_schedule = ["ddg", "alg"]

    ddg_iter = cycle(ddg_train)
    alg_iter = cycle(alg_train)

    # For a fair "epoch", define steps = max(len(ddg_train), len(alg_train)) * len(schedule)
    steps_per_epoch = max(len(ddg_train), len(alg_train)) * len(task_schedule)

    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        model.esm.train(cfg.get("unfreeze_last_n", 0) > 0)

        ddg_sse, ddg_n = 0.0, 0
        pbar = tqdm(range(steps_per_epoch), desc=f"Epoch {epoch}/{cfg['epochs']}", leave=False)

        for step in pbar:
            task = task_schedule[step % len(task_schedule)]
            opt.zero_grad(set_to_none=True)

            if task == "ddg":
                wt, mut, y, _ = next(ddg_iter)
                wt, mut, y = wt.to(DEVICE), mut.to(DEVICE), y.to(DEVICE)
                pred = model.forward_ddg(wt, mut)
                loss = cfg.get("lambda_ddg", 1.0) * mse(pred, y)

                # accumulate ddg rmse stats
                bs = y.numel()
                ddg_sse += float(loss.detach().cpu()) * bs / max(cfg.get("lambda_ddg", 1.0), 1e-12)
                ddg_n   += bs

                loss.backward()

            else:  # "alg"
                tok, y, _ = next(alg_iter)
                tok, y = tok.to(DEVICE), y.to(DEVICE)
                logits = model.forward_alg(tok)
                loss = cfg.get("lambda_alg", 1.0) * bce(logits, y)
                loss.backward()

            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.get("grad_clip", 1.0))
            opt.step()

            pbar.set_postfix({"task": task, "loss": f"{float(loss.detach().cpu()):.4f}"})

        # ── Validate ΔΔG ─────────────────────────────────────
        model.eval()
        ddg_preds, ddg_trues = [], []
        with torch.no_grad():
            for wt, mut, y, _ in ddg_val:
                wt, mut = wt.to(DEVICE), mut.to(DEVICE)
                ddg_preds.append(model.forward_ddg(wt, mut).cpu().numpy())
                ddg_trues.append(y.numpy())
        p = np.concatenate(ddg_preds); t = np.concatenate(ddg_trues)

        ddg_train_rmse = math.sqrt(ddg_sse / max(ddg_n, 1))
        ddg_val_rmse   = float(np.sqrt(np.mean((p - t) ** 2)))
        r              = float(pearsonr(p, t)[0]) if len(p) > 1 else float("nan")
        rho            = float(spearmanr(p, t)[0]) if len(p) > 1 else float("nan")

        # ── Validate Allergen ────────────────────────────────
        alg_probs, alg_true = [], []
        with torch.no_grad():
            for tok, y, _ in alg_val:
                tok = tok.to(DEVICE)
                logits = model.forward_alg(tok).cpu()
                probs  = torch.sigmoid(logits).numpy()
                alg_probs.append(probs)
                alg_true.append(y.numpy())
        alg_probs = np.concatenate(alg_probs)
        alg_true  = np.concatenate(alg_true)

        if _HAS_SKLEARN:
            alg_auc = float(roc_auc_score(alg_true, alg_probs)) if len(np.unique(alg_true)) > 1 else float("nan")
            alg_acc = float(accuracy_score(alg_true, (alg_probs >= 0.5).astype(int)))
        else:
            # fallback: only accuracy-like metric
            alg_auc = float("nan")
            alg_acc = float(np.mean((alg_probs >= 0.5) == alg_true))

        scheduler.step(ddg_val_rmse)
        lr_head = opt.param_groups[0]["lr"]
        lr_esm  = opt.param_groups[1]["lr"]

        history["ddg_train_rmse"].append(ddg_train_rmse)
        history["ddg_val_rmse"].append(ddg_val_rmse)
        history["ddg_pearson"].append(r)
        history["ddg_spearman"].append(rho)
        history["alg_val_auc"].append(alg_auc)
        history["alg_val_acc"].append(alg_acc)

        # Save history every epoch
        torch.save(history, os.path.join(save_dir, f"history_epoch_{epoch}.pth"))

        print(
            f"Epoch {epoch:2d} | "
            f"ΔΔG Train RMSE: {ddg_train_rmse:.4f} | ΔΔG Val RMSE: {ddg_val_rmse:.4f} | r: {r:.3f} | ρ: {rho:.3f} | "
            f"Alg AUC: {alg_auc:.3f} | Alg Acc: {alg_acc:.3f} | "
            f"lr_head: {lr_head:.2e} | lr_esm: {lr_esm:.2e}"
        )

        # Early-stop on ΔΔG RMSE (you can change this if allergen is primary)
        if ddg_val_rmse < best_rmse - 1e-4:
            best_rmse, patience_left = ddg_val_rmse, cfg["patience"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

            checkpoint = {
                "epoch": epoch,
                "best_ddg_val_rmse": best_rmse,
                "cfg": cfg,
                "model_state": best_state,
                "optimizer_state": opt.state_dict(),
                "scheduler_state": scheduler.state_dict(),
            }
            torch.save(checkpoint, best_path)
            print(f"  → Saved best checkpoint to: {best_path}")
        else:
            patience_left -= 1
            if patience_left == 0:
                print(f"Early stopping. Best ΔΔG Val RMSE: {best_rmse:.4f}")
                break

    model.load_state_dict(best_state)
    return model, alphabet, history, best_path, best_rmse, save_dir


# ── Simple plots ────────────────────────────────────────────
def plot_history_multitask(history, save_dir="."):
    epochs = range(1, len(history["ddg_val_rmse"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

    axes[0].plot(epochs, history["ddg_train_rmse"], label="ΔΔG Train RMSE")
    axes[0].plot(epochs, history["ddg_val_rmse"],   label="ΔΔG Val RMSE")
    axes[0].set(xlabel="Epoch", ylabel="RMSE (kcal/mol)", title="ΔΔG Learning Curves")
    axes[0].legend(frameon=False)

    axes[1].plot(epochs, history["alg_val_acc"], label="Alg Val Acc")
    if not all(np.isnan(history["alg_val_auc"])):
        axes[1].plot(epochs, history["alg_val_auc"], label="Alg Val AUC")
    axes[1].set(xlabel="Epoch", ylabel="Metric", title="Allergen Validation")
    axes[1].legend(frameon=False)

    for ax in axes:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "multitask_history.pdf"), bbox_inches="tight")
    plt.show()


# ── Run sweep on unfreeze_last_n ─────────────────────────────

sweep = [2]
results = []
all_histories = {}

for n in sweep:
    cfg_run = dict(CFG)
    cfg_run["unfreeze_last_n"] = n

    print(f"\n=== Multi-task: unfreeze_last_n = {n} ===")
    model, alphabet, history, best_path, best_rmse, save_dir = train_multitask(cfg_run)

    plot_history_multitask(history, save_dir=save_dir)
    results.append((n, best_rmse, best_path))
    all_histories[n] = history

print("\nSummary (rank by best ΔΔG Val RMSE):")
for n, rmse, path in sorted(results, key=lambda x: x[1]):
    print(f"unfreeze={n} | best_ddg_val_rmse={rmse:.4f} | {path}")

Device: cuda

=== Multi-task: unfreeze_last_n = 2 ===


FileNotFoundError: [Errno 2] No such file or directory: 'project_data/mega_train.csv'

In [3]:
import pandas as pd

df = pd.read_csv("project_data/algpred2_train_seq.csv")
print(df.columns.tolist())
print(df.head(2))

['id', 'sequence', 'label']
     id                                           sequence  label
0  P_13  MGKPFTLSLSSLCLLLLSSACFAISSSKLNECQLNNLNALEPDHRV...      1
1  P_14  MGVFTFEDEINSPVAPATLYKALVTDADNVIPKALDSFKSVENVEG...      1
